### Wi-Fi Sensing

#### Initial Setup

In [1]:
# pandas for data manipulation
# re for regular expressions
import re

# pyplot for plotting
import matplotlib.pyplot as plt

# numpy for numerical operations
import numpy as np
import pandas as pd

# seaborn for advanced plotting
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, make_scorer

# sklearn for machine learning
from sklearn.model_selection import GridSearchCV, train_test_split

# other .py files
from utils.csv_import import get_csv_files_generalistic, sort_meta_info

In [2]:
# Retrieve CSV files
path: str = (
	"C:\\Users\\pedro\\OneDrive - Universidade de Coimbra\\Ambiente de Trabalho"
	"\\tese\\thesis-project\\data\\CSI DATA DM RENAMED [com w00 alterado]"
)

# path: str = "C:\\Users\\Oscar\\Documents\\GitHub\\thesis-project_isac\\CSI DATA RENAMED [alterado]"

FileMap = dict[str, dict[str, str]]
csi_map = dict[str, dict[str, np.ndarray]]

# data_files["user_1"]["a09"]["esp_1"]
# data_files["user_0"]["z00"]["esp_1"][0]
data_files = get_csv_files_generalistic(path)
users_id, posicoes, esp_ids, repetition_ids = sort_meta_info(path)

# Initialize variables
subcarriers: list[int] = list(range(51))

#### *Def functions*

##### Print functions info

In [3]:
def info(file: FileMap) -> None:
	# Print information about the files dictionaries
	print("Número de ficheiros recolhidos: ", len(file))

	# avoid StopIteration if dict is empty
	esp_count = len(next(iter(file.values()))) if file else 0
	print("ESPs por cenário:", esp_count)

	# Print all scenarios names
	print("\nTodos os cenários:", sorted(file.keys()))

	# Count scenarios by letter prefix
	scenario_prefixes: dict[str, int] = {}
	for scenario in file:
		prefix = scenario[0]
		scenario_prefixes[prefix] = scenario_prefixes.get(prefix, 0) + 1

	print("\nN.º cenários/posição:")
	for prefix, count in sorted(scenario_prefixes.items()):
		print(f"Posição {prefix}: {count} cenários")

	print("---\n")

##### Cálculo módulo (magnitude) ((posição x esp) (array de arrays))

In [4]:
# função que processo o arquivo CSV completo
# # e extrai a info da coluna do CSI
# # converte em números complexos
# # calcula a magnitude (2.º passo)
# #
# e faz ainda uma 1a limpeza
# # remove 2 primeiras subportadoras
# # remove colunas a zero
# # aplica FFT shift

def process_csi(file: str) -> np.ndarray:

	df = pd.read_csv(file, header=None)
	# file read

	# no da diana são coletadas 120 amostras
	csi_raw: pd.Series = df.iloc[:, 26]

	total_sc: int = 128
	valid_csi: list[list[float]] = []

	# fiquei aqui 19/11
	for entry in csi_raw:
		match = re.search(r"\[(.*?)\]", str(entry))
		if not match:
			continue
		nums = [float(n) for n in re.findall(r"-?\d+", match.group(1))]
		if len(nums) == total_sc:
			valid_csi.append(nums)

	valid_csi = np.array(valid_csi)
	complex_csi = valid_csi[:, ::2] + 1j * valid_csi[:, 1::2]
	magnitudes = np.abs(complex_csi)

	# Limpeza: remover 2 primeiras subportadoras e colunas todas-zero
	magnitudes = magnitudes[:, 2:]
	magnitudes = magnitudes[:, ~np.all(magnitudes == 0, axis=0)]
	magnitudes = np.fft.fftshift(magnitudes, axes=1)

	return magnitudes


# cria o dict magnitudes
# itera sobre cada posicao da grelha
# # para cada posicao, itera sobre cada esp_id e path
# # devolve o dict magnitudes preenchido
def process_magnitudes(files: FileMap) -> dict[str, dict[str, np.ndarray]]:
	magnitudes: dict[str, dict[str, dict[str, np.ndarray]]] = {}

	for user_key, posicoes_map in files.items():
		magnitudes[user_key] = {}

		for posicao, esps_map in posicoes_map.items():
			magnitudes[user_key][posicao] = {}

			for esp_key, file in esps_map.items():
				if file is not None:
					magnitudes[user_key][posicao][esp_key] = process_csi(file)

	return magnitudes


# o dict magnitudes fica
# posicao1
# # esp1
# # # array de arrays (cada leitura é um array)
# # ...
# # esp4
# # # array de arrays (cada leitura é um array)

##### Cálculo da média

In [ ]:
# função que recebe o dicionário com a posição a ser utilizada
# # e calcula a média para cada subportadora (coluna a coluna)
# # para cada esp
def calc_mean_per_subcarriers(magnitudes_posicao: csi_map, frac: float) -> dict[int, dict[str, np.ndarray]]:

	media: dict[int, np.ndarray] = {}
	amostras: int = 0

	# número de amostras a recolher = n.º linhas * frac (= Miguel)
	for esp_id in esp_ids:
		esp_key = f"esp_{esp_id}"
		amostras = int(magnitudes_posicao[esp_key].shape[0] * frac)
		media[esp_key] = np.mean(magnitudes_posicao[esp_key][:amostras, :], axis=0)
	return media

In [ ]:
# ---------- USER 1 -------------------- USER 2
# ---Mean----Std----Max----------Mean----Std----Max---
# esp1-esp2-esp3...-------------esp1-esp2...-esp1-esp2...
# sc1 sc2 sc3 ...--------------sc1 sc2 ... sc1 sc2 ...

def define_X(magnitudes):

	df_X: list[np.ndarray] = []

	for user_id in users_id:


		# chamar função para pipeline

	return df_X


##### Criar dataset "dados" (sc x posição) (array)

In [6]:
# para cada posição
# # ver o tamanho de cada (auto definido em cima)
# # e para cada subportadora
# # # e para cada esp
# # # # extrair o vetor correto
# # # # guardar no dicionário

dados_map = dict[int, dict[str, np.ndarray]]


def criar_dataset(
	posicoes: list[str], tamanhos: dict[str, int], magnitudes: csi_map
) -> dados_map:
	dados: dados_map = {}
	dados = {sc: {} for sc in subcarriers}

	for posicao in posicoes:
		tamanho = tamanhos[posicao]
		for sc in subcarriers:
			for esp_id in esps_id:
				# acede ao array de arrays
				matriz = magnitudes[posicao][f"esp_{esp_id}"]
				# extrai o vetor coluna
				# com (linhas/leituras = tamanho)
				# com (coluna = subc)
				vetor = matriz[:tamanho, sc]
				# print("nº linhas = ", matriz.shape[0], " e tamanho = ", tamanho)
				chave = f"{posicao}_esp_{esp_id}"
				dados[sc][chave] = vetor

	return dados

##### Normalizar dados

In [7]:
# Função para normalizar
# a função recebe o dicionário de vetores e o dicionário de médias
# e devolve o dicionário de vetores normalizados
# cada vetor é normalizado subtraindo a média e dividindo pela média
def normalizar_vetores(
	posicoes: list[str],
	dados: dados_map,
	medias: dict[int, dict[str, float]],
) -> dados_map:
	normalizados: dados_map = {}
	normalizados = {sc: {} for sc in subcarriers}

	for sc in subcarriers:
		for posicao in posicoes:
			for esp_id in esps_id:
				chave = f"{posicao}_esp_{esp_id}"
				vetor = dados[sc][chave]

				esp = f"esp_{esp_id}"
				media = medias[sc][esp]

				normalizados[sc][chave] = (vetor - media) / media
	return normalizados


# Função para normalizar dados a afinar
def normalizar_vetores_afinar(
	posicoes_afinar: list[str],
	vetores: dados_map,
	medias: dict[int, dict[str, float]],
) -> dados_map:
	normalizados: dados_map = {}
	normalizados = {sc: {} for sc in subcarriers}

	for sc in subcarriers:
		for posicao in posicoes_afinar:
			for esp_id in esps_id:
				chave = f"{posicao}_esp_{esp_id}"
				vetor = vetores[sc][chave]

				esp = f"esp_{esp_id}"
				media = medias[sc][esp]

				normalizados[sc][chave] = (vetor - media) / media
	return normalizados

##### Cálculo mean, std, máx

In [8]:
# função para calcular a média em janelas sobrepostas
def media_grupos_overlap(
	vetor: np.ndarray, tamanho_janela: int, step: int
) -> np.ndarray:
	medias: list[float] = []
	for i in range(0, len(vetor) - tamanho_janela + 1, step):
		janela = vetor[i : i + tamanho_janela]
		medias.append(float(np.mean(janela)))
	return np.array(medias)


# função para calcular o desvio padrão em janelas sobrepostas
def std_grupos_overlap(vetor: np.ndarray, tamanho_janela: int, step: int) -> np.ndarray:
	desvios: list[float] = []
	for i in range(0, len(vetor) - tamanho_janela + 1, step):
		janela = vetor[i : i + tamanho_janela]
		desvios.append(float(np.std(janela)))
	return np.array(desvios)


# função para calcular o máximo em janelas sobrepostas,
# evitando repetições dos últimos 3 máximos escolhidos
def max_grupos_overlap(vetor: np.ndarray, tamanho_janela: int, step: int) -> np.ndarray:
	maximos: list[float] = []
	for i in range(0, len(vetor) - tamanho_janela + 1, step):
		janela = vetor[i : i + tamanho_janela]
		candidatos = np.sort(np.unique(janela))[::-1]  # do maior para o menor

		# Exclui os 3 últimos valores
		ultimos = set(maximos[-3:])

		escolhido = None
		for val in candidatos:
			if val not in ultimos:
				escolhido = val
				break

		# Se todos os valores estão nos últimos 3, aceita o maior mesmo assim
		if escolhido is None:
			escolhido = candidatos[0]

		maximos.append(escolhido)

	return np.array(maximos)


def media_fixed(vetor: np.ndarray, tamanho_janela: int, w: int) -> np.ndarray:
	step = (len(vetor) - tamanho_janela) // (w - 1)
	return media_grupos_overlap(vetor, tamanho_janela, step)[:w]


def std_fixed(vetor: np.ndarray, tamanho_janela: int, w: int) -> np.ndarray:
	step = (len(vetor) - tamanho_janela) // (w - 1)
	return std_grupos_overlap(vetor, tamanho_janela, step)[:w]


def max_fixed(vetor: np.ndarray, tamanho_janela: int, w: int) -> np.ndarray:
	step = (len(vetor) - tamanho_janela) // (w - 1)
	return max_grupos_overlap(vetor, tamanho_janela, step)[:w]

#### *Pre Processing*

Módulo

In [9]:
def count_processed_arrays(magnitudes) -> int:

    array_count = 0

    # Nível 1: Chave de utilizador (user_key)
    for user_key, posicoes_map in magnitudes.items():

        # Nível 2: Posição (posicao)
        for posicao, esps_map in posicoes_map.items():

            # Nível 3: ESP (esp_key)
            for esp_key, data_array in esps_map.items():

                # 1. Verifica se o objeto é um array NumPy
                if isinstance(data_array, np.ndarray):

                    # 2. Verifica se o array não está vazio (ou seja, se foi processado com dados)
                    if data_array.size > 0:
                        array_count += 1
                        print(f"  [ENCONTRADO] Array em: {user_key} -> {posicao} -> {esp_key} (Tamanho: {data_array.size})")
                    else:
                        print(f"  [VAZIO] Array em: {user_key} -> {posicao} -> {esp_key} (Sem dados)")
                else:
                    # Caso de erro (não deve acontecer se a MagnitudesMap for bem construída)
                    print(f"  [ERRO] Valor inesperado em: {user_key} -> {posicao} -> {esp_key} (Tipo: {type(data_array)})")

    return array_count

In [10]:
# info(data_files)

magnitudes: csi_map = process_magnitudes(data_files)

In [ ]:
print("número de arrays processados: ", count_processed_arrays(magnitudes))
magnitudes['user_2']['d11']['esp_2'].shape

Média para normalizar

In [ ]:
# posição para retirar normalizações - escolha ARBITRÁRIA (explorar outras opções)
# vou manter, mas futuramente alterarei - fica já renomeado
posicao: str = "c06"

# fração dos valores a considerar - porquê este valor?
# também vou manter e estudo a seguir este valor
frac: float = 1 / 6

mean_ref: dict[str, dict[int, np.ndarray]] = {}

for user_id in users_id:
	user_key = f'user_{user_id}'
	if magnitudes[user_key][f'{posicao}'] != {}:
		mean_ref[user_key] = calc_mean_per_subcarriers(magnitudes[user_key][f'{posicao}'], frac)

Dataset "dados"

In [ ]:
# Parâmetros

# tamanhos fixos selecionados para quando
# se fizer o "step" ficar número certo de amostras
#
# sem pessoa tem mais dados
N = 90
M = 350
O = 60
A = 292
I = 200

tamanhos_loureiro: dict[str, int] = {}
tamanhos_diana: dict[str, int] = {}
tamanhos_afinar: dict[str, int] = {}

tamanhos = {
	"vazio_1": M,
	"vazio_2": M,
	"a00": N,
	"c05": N,
	"a01": N,
	"c06": N,
	"a09": N,
	"c07": N,
	"a10": N,
	"c08": N,
	"a11": N,
	"c09": N,
	"b00": N,
	"c10": N,
	"b01": N,
	"c11": N,
	"b02": N,
	"d01": N,
	"b03": N,
	"d02": N,
	"b04": N,
	"d03": N,
	"b05": N,
	"d04": N,
	"b06": N,
	"d05": N,
	"b07": N,
	"d06": N,
	"b08": N,
	"d07": N,
	"b09": N,
	"d08": N,
	"b10": N,
	"d09": N,
	"b11": N,
	"d10": N,
	"c01": N,
	"d11": N,
	"c02": N,
	"e04": N,
	"c03": N,
	"e05": N,
	"c04": N,
	"e06": N,
}

tamanhos_diana = {
	"vazio_1": M,
	"vazio_2": M,
	"a00": O,
	"c05": O,
	"a01": O,
	"c06": O,
	"a09": O,
	"c07": O,
	"a10": O,
	"c08": O,
	"a11": O,
	"c09": O,
	"b00": O,
	"c10": O,
	"b01": O,
	"c11": O,
	"b02": O,
	"d01": O,
	"b03": O,
	"d02": O,
	"b04": O,
	"d03": O,
	"b05": O,
	"d04": O,
	"b06": O,
	"d05": O,
	"b07": O,
	"d06": O,
	"b08": O,
	"d07": O,
	"b09": O,
	"d08": O,
	"b10": O,
	"d09": O,
	"b11": O,
	"d10": O,
	"c01": O,
	"d11": O,
	"c02": O,
	"e04": O,
	"c03": O,
	"e05": O,
	"c04": O,
	"e06": O,
}

tamanhos_afinar = {
	"com_1": A,
	"sem_1": A,
	"com_2": I,
	"sem_2": I,
}

posicoes: list[str] = []
posicoes_afinar: list[str] = []

# do Miguel não há a02
posicoes = [
	"vazio_1",
	"vazio_2",
	"a00",
	"a01",
	"a09",
	"a10",
	"a11",
	"b00",
	"b01",
	"b02",
	"b03",
	"b04",
	"b05",
	"b06",
	"b07",
	"b08",
	"b09",
	"b10",
	"b11",
	"c01",
	"c02",
	"c03",
	"c04",
	"c05",
	"c06",
	"c07",
	"c08",
	"c09",
	"c10",
	"c11",
	"d01",
	"d02",
	"d03",
	"d04",
	"d05",
	"d06",
	"d07",
	"d08",
	"d09",
	"d10",
	"d11",
	"e04",
	"e05",
	"e06",
]

posicoes_afinar = ["sem_1", "sem_2", "com_1", "com_2"]

In [ ]:
for sc in subcarriers:
	print(f"Subportadora {sc}:")
	for posicao in posicoes:
		for esp_id in esps_id:
			chave = f"{posicao}_esp_{esp_id}"
			tamanho_vetor = magnitudes_loureiro[posicao][f"esp_{esp_id}"].shape[0]
			esperado = tamanhos[posicao]
			print(
				f"  Verificando {chave}: tamanho vetor = {tamanho_vetor}, esperado = {esperado}"
			)

In [ ]:
dados_loureiro: dados_map = {}
dados_diana: dados_map = {}
dados_afinar: dados_map = {}

dados_loureiro = criar_dataset(posicoes, tamanhos, magnitudes_loureiro)
dados_diana = criar_dataset(posicoes, tamanhos_diana, magnitudes_diana)
dados_afinar = criar_dataset(posicoes_afinar, tamanhos_afinar, magnitudes_afinadas)

In [ ]:
size_vazio: int = 0
size_com: int = 0

for sc in subcarriers:
	for posicao in posicoes:
		for esp_id in esps_id:
			if posicao.startswith("vazio"):
				size_vazio += dados_loureiro[sc][f"{posicao}_esp_{esp_id}"].shape[0]
			else:
				size_com += dados_loureiro[sc][f"{posicao}_esp_{esp_id}"].shape[0]

print(f"Tamanho vetores vazio: {size_vazio}")
print(f"Tamanho vetores com pessoa: {size_com}")

In [ ]:
print(magnitudes_loureiro["a00"]["esp_1"].shape)
print(dados_loureiro[subcarriers[0]]["a00_esp_1"].shape)

print(magnitudes_loureiro["c06"]["esp_2"].shape)
print(dados_loureiro[subcarriers[0]]["c06_esp_2"].shape)

print(magnitudes_loureiro["vazio_1"]["esp_1"].shape)
print(dados_loureiro[subcarriers[0]]["vazio_1_esp_1"].shape)

Normalização de dados "normalizados"

In [ ]:
normalizados_loureiro: dados_map = {}
normalizados_diana: dados_map = {}
normalizados_afinar: dados_map = {}

normalizados_loureiro = normalizar_vetores(posicoes, dados_loureiro, medias_c06)
normalizados_diana = normalizar_vetores(posicoes, dados_diana, medias_c06)
normalizados_afinar = normalizar_vetores_afinar(
	posicoes_afinar, dados_afinar, medias_c06
)

Mean, std, max

In [ ]:
tamanho_janela: int = 10
step: int = 3
W: int = 27  # nº de janelas fixas para "com pessoa"
G: int = 60  # nº de janelas fixas para "dados_afinar"
# Estes valores diferentes são estranhos

media_loureiro: dados_map = {sc: {} for sc in subcarriers}
media_diana: dados_map = {sc: {} for sc in subcarriers}
media_afinar: dados_map = {sc: {} for sc in subcarriers}

std_loureiro: dados_map = {sc: {} for sc in subcarriers}
std_diana: dados_map = {sc: {} for sc in subcarriers}
std_afinar: dados_map = {sc: {} for sc in subcarriers}

max_loureiro: dados_map = {sc: {} for sc in subcarriers}
max_diana: dados_map = {sc: {} for sc in subcarriers}
max_afinar: dados_map = {sc: {} for sc in subcarriers}

for posicao in posicoes:
	for esp_id in esps_id:
		chave = f"{posicao}_esp_{esp_id}"
		for sc in subcarriers:
			valores_l = normalizados_loureiro[sc][chave]
			valores_d = normalizados_diana[sc][chave]

			# Porque este step? Não sei o impacto final, mas isto parece-me batota...
			if posicao.startswith("vazio"):
				# mantém todas as janelas possíveis, sem truncar
				media_l = media_grupos_overlap(valores_l, tamanho_janela, step)
				std_l = std_grupos_overlap(valores_l, tamanho_janela, step)
				m_l = max_grupos_overlap(valores_l, tamanho_janela, step)

				media_d = media_grupos_overlap(valores_d, tamanho_janela, step)
				std_d = std_grupos_overlap(valores_d, tamanho_janela, step)
				m_d = max_grupos_overlap(valores_d, tamanho_janela, step)

			else:
				# força W janelas idênticas
				media_l = media_fixed(valores_l, tamanho_janela, W)
				std_l = std_fixed(valores_l, tamanho_janela, W)
				m_l = max_fixed(valores_l, tamanho_janela, W)

				media_d = media_fixed(valores_d, tamanho_janela, W)
				std_d = std_fixed(valores_d, tamanho_janela, W)
				m_d = max_fixed(valores_d, tamanho_janela, W)

			media_loureiro[sc][chave] = media_l
			std_loureiro[sc][chave] = std_l
			max_loureiro[sc][chave] = m_l

			media_diana[sc][chave] = media_d
			std_diana[sc][chave] = std_d
			max_diana[sc][chave] = m_d

for posicao in posicoes_afinar:
	for esp_id in esps_id:
		chave = f"{posicao}_esp_{esp_id}"
		for sc in subcarriers:
			v = normalizados_afinar[sc][chave]
			media = media_fixed(v, tamanho_janela, G)
			std = std_fixed(v, tamanho_janela, G)
			max = max_fixed(v, tamanho_janela, G)

			media_afinar[sc][chave] = media
			std_afinar[sc][chave] = std
			max_afinar[sc][chave] = max

#### *ML Pre Processing*

##### Tabela colunas

In [ ]:
subcarriers_map: dict[int, int] = {i + 1: sc for i, sc in enumerate(subcarriers)}

# dicionários para guardar cada coluna
coluna_media: dict[str, np.ndarray] = {}
coluna_std: dict[str, np.ndarray] = {}
coluna_max: dict[str, np.ndarray] = {}

# dicionários para guardar cada coluna
coluna_media_afinar: dict[str, np.ndarray] = {}
coluna_std_afinar: dict[str, np.ndarray] = {}
coluna_max_afinar: dict[str, np.ndarray] = {}

for sc_label, sc in subcarriers_map.items():
	for esp_id in esps_id:
		# nome da coluna
		name_mean = f"Mean sc_{sc_label} esp_{esp_id}"
		name_std = f"Std sc_{sc_label} esp_{esp_id}"
		name_max = f"Max sc_{sc_label} esp_{esp_id}"

		sequencia_mean, sequencia_std, sequencia_max = [], [], []

		# 1) SEM PESSOA: primeiro Loureiro, depois Diana
		sequencia_mean.append(media_loureiro[sc][f"vazio_1_esp_{esp_id}"])
		sequencia_mean.append(media_loureiro[sc][f"vazio_2_esp_{esp_id}"])

		sequencia_mean.append(media_diana[sc][f"vazio_1_esp_{esp_id}"])
		sequencia_mean.append(media_diana[sc][f"vazio_2_esp_{esp_id}"])

		sequencia_std.append(std_loureiro[sc][f"vazio_1_esp_{esp_id}"])
		sequencia_std.append(std_loureiro[sc][f"vazio_2_esp_{esp_id}"])

		sequencia_std.append(std_diana[sc][f"vazio_1_esp_{esp_id}"])
		sequencia_std.append(std_diana[sc][f"vazio_2_esp_{esp_id}"])

		sequencia_max.append(max_loureiro[sc][f"vazio_1_esp_{esp_id}"])
		sequencia_max.append(max_loureiro[sc][f"vazio_2_esp_{esp_id}"])

		sequencia_max.append(max_diana[sc][f"vazio_1_esp_{esp_id}"])
		sequencia_max.append(max_diana[sc][f"vazio_2_esp_{esp_id}"])

		# 2) COM PESSOA: percorre todas as outras posições

		# GRALHA CÓDIGO ORIGINAL
		# # o "if" apenas serve para o grupo "vazio_1" e não para "vazio_2"
		for posicao in posicoes:
			if posicao.startswith("vazio"):
				continue

			key = f"{posicao}_esp_{esp_id}"
			sequencia_mean.append(media_loureiro[sc][key])
			sequencia_mean.append(media_diana[sc][key])

			sequencia_std.append(std_loureiro[sc][key])
			sequencia_std.append(std_diana[sc][key])

			sequencia_max.append(max_loureiro[sc][key])
			sequencia_max.append(max_diana[sc][key])

		# 3) concatena e guarda
		coluna_media[name_mean] = np.concatenate(sequencia_mean)
		coluna_std[name_std] = np.concatenate(sequencia_std)
		coluna_max[name_max] = np.concatenate(sequencia_max)

for sc_label, sc in subcarriers_map.items():
	for esp_id in esps_id:
		# nome da coluna
		name_mean_af = f"Mean sc_{sc_label} esp_{esp_id}"
		name_std_af = f"Std sc_{sc_label} esp_{esp_id}"
		name_max_af = f"Max sc_{sc_label} esp_{esp_id}"

		seq_mean, seq_std, seq_max = [], [], []

		for posicao in posicoes_afinar:
			key = f"{posicao}_esp_{esp_id}"

			seq_mean.append(media_afinar[sc][key])
			seq_std.append(std_afinar[sc][key])
			seq_max.append(max_afinar[sc][key])

		coluna_media_afinar[name_mean_af] = np.concatenate(seq_mean)
		coluna_std_afinar[name_std_af] = np.concatenate(seq_std)
		coluna_max_afinar[name_max_af] = np.concatenate(seq_max)

In [ ]:
tamanho_loureiro = (
	len(media_loureiro[1]["vazio_1_esp_1"])
	+ len(media_loureiro[1]["vazio_1_esp_1"])
	+ len(media_loureiro[1]["a00_esp_1"]) * 42
)
print(len(media_loureiro[1]["vazio_1_esp_1"]) + len(media_loureiro[1]["vazio_1_esp_1"]))
print(tamanho_loureiro)

tamanhos_diana = (
	len(media_diana[1]["vazio_1_esp_1"])
	+ len(media_diana[1]["vazio_1_esp_1"])
	+ len(media_diana[1]["a00_esp_1"]) * 42
)
print(len(media_diana[1]["vazio_1_esp_1"]) + len(media_diana[1]["vazio_1_esp_1"]))
print(tamanhos_diana)

total = tamanho_loureiro + tamanhos_diana
print(total)
print(len(coluna_media["Mean sc_1 esp_1"]))

##### Coluna target

In [ ]:
# truncar todas as colunas numa só
# para verificar qual o tamanho do menor vetor
all_cols: dict[str, np.ndarray] = {**coluna_media, **coluna_std, **coluna_max}
min_len: int = min(len(v) for v in all_cols.values())

# trunca as colunas para o tamanho do menor vetor (min_len)
coluna_media = {k: v[:min_len] for k, v in coluna_media.items()}
coluna_std = {k: v[:min_len] for k, v in coluna_std.items()}
coluna_max = {k: v[:min_len] for k, v in coluna_max.items()}

# Gera o vetor de labels (0 = vazio, 1 = com_pessoa)
# o número de zeros é a soma dos tamanhos dos grupos "sem pessoa"
# tanto do Loureiro como da Diana
# # a subportadora 22 é escolhida arbitrariamente, pois todas têm o mesmo número de leituras
n_zero: int = sum(
	len(media_loureiro[22][f"vazio_1_esp_{esp_id}"])
	+ len(media_loureiro[22][f"vazio_2_esp_{esp_id}"])
	+ len(media_diana[22][f"vazio_1_esp_{esp_id}"])
	+ len(media_diana[22][f"vazio_2_esp_{esp_id}"])
	for esp_id in esps_id
)

# Total de linhas (após truncar tudo para min_len)
n_total: int = min_len

# Cria vetor target: 0 para "sem pessoa", 1 para "com pessoa"
target_vector: np.ndarray = np.zeros(n_total, dtype=int)
target_vector[n_zero:] = 1  # a partir do índice n_zero, coloca 1 (com pessoa)

Criar Dataframe

In [ ]:
# DataFrame
df: pd.DataFrame = pd.DataFrame({**coluna_media, **coluna_std, **coluna_max})
df["target"] = target_vector

print(df["target"].value_counts())
print(df.shape)

# descrever categorical columns
df.describe()

#### *Machine Learning*

##### Dataframe normal

Divisão treino/teste

In [ ]:
# features
X: np.ndarray = df.drop(columns="target").to_numpy()

# target
y = df["target"].to_numpy()

print("X.shape =", X.shape)
print("y.shape =", y.shape)


X_train, X_test, y_train, y_test = train_test_split(
	X,
	y,
	test_size=0.2,
	random_state=200,
)

print("X_train.shape =", X_train.shape)
print("X_test.shape  =", X_test.shape)
print("y_train.shape =", y_train.shape)
print("y_test.shape  =", y_test.shape)


Random Forest Classifier

In [ ]:
model = RandomForestClassifier(
	min_samples_split=4,
	min_samples_leaf=4,
	max_leaf_nodes=64,
	n_estimators=150,
	random_state=200,
	max_depth=16,
	class_weight="balanced_subsample",
	criterion="entropy",
)

model = model.fit(X_train, y_train)

# pkl_path = path + "\\modelo_treinado.pkl"
# joblib.dump(model, pkl_path)
# print(f"Modelo guardado em: {pkl_path}")

In [ ]:
pred_test = model.predict(X_test)

bal_acc = balanced_accuracy_score(y_test, pred_test)
print("Balanced accuracy:", bal_acc * 100, "%")

Confusion Matrix

In [ ]:
confusion_matrix_test = confusion_matrix(y_test, pred_test)

In [ ]:
sns.set_theme(font_scale=1.2)
plt.figure(figsize=(6, 5))

sns.heatmap(
	confusion_matrix_test,
	annot=True,
	fmt="d",
	cmap="Reds",
	xticklabels=["Sem pessoa", "Com pessoa"],
	yticklabels=["Sem pessoa", "Com pessoa"],
)

plt.title("Matriz de Confusão")
plt.xlabel("Previsto")
plt.ylabel("Real")
plt.tight_layout()
plt.show()


##### Dataframe Afinar

Criar Dataframe

In [ ]:
# verificar menor tamanho das colunas
all_cols_afinar = {**coluna_media_afinar, **coluna_std_afinar, **coluna_max_afinar}
min_len_af: int = min(len(v) for v in all_cols_afinar.values())

# trunca todas as colunas ao mesmo tamanho
coluna_media_afinar = {k: v[:min_len_af] for k, v in coluna_media_afinar.items()}
coluna_std_afinar = {k: v[:min_len_af] for k, v in coluna_std_afinar.items()}
coluna_max_afinar = {k: v[:min_len_af] for k, v in coluna_max_afinar.items()}

# Do Miguel:
# # Como há apenas 4 ESPs x 1 ficheiro por classe, a divisão é simples:
# # Cada classe (com_pessoa, vazio) tem 4 séries → total 8
# # O label muda a meio

# Total de linhas após truncamento
n_total_afinar = min_len_af
n_zero_afinar = n_total_afinar // 2  # metade "sem pessoa", metade "com pessoa"

# Gera os rótulos
target_vector_afinar = np.zeros(n_total_afinar, dtype=int)
target_vector_afinar[n_zero_afinar:] = 1

# DF Afinar
df_afinar = pd.DataFrame(
	{**coluna_media_afinar, **coluna_std_afinar, **coluna_max_afinar}
)
df_afinar["label"] = target_vector_afinar

# Verifica
print(df_afinar["label"].value_counts())
print(df_afinar.shape)
display(df_afinar)

Confusion Matrix

In [ ]:
# Garantir arrays numpy (evita ExtensionArray / typing issues)
X_afinar: np.ndarray = df_afinar.drop(columns="label").to_numpy()
y_afinar: np.ndarray = df_afinar["label"].to_numpy()

# Prever com o modelo atual
y_pred_afinar = model.predict(X_afinar)

# Índices dos erros
falsos_negativos = (y_afinar == 1) & (y_pred_afinar == 0)
falsos_positivos = (y_afinar == 0) & (y_pred_afinar == 1)

# Extrair exemplos incorretamente classificados
X_erros = X_afinar[falsos_negativos | falsos_positivos]
y_erros = y_afinar[falsos_negativos | falsos_positivos]

print(f"Total de exemplos a adicionar (FP + FN): {len(y_erros)}")

# --- Calcular a matriz de confusão ---
cm = confusion_matrix(y_afinar, y_pred_afinar)
print("Confusion Matrix (dataset2):")
print(cm)

In [ ]:
# Reforçar treino com os exemplos mal classificados
X_treinado_reforcado = np.vstack([X_train, X_erros])
y_treinado_reforcado = np.hstack([y_train, y_erros])

print(X_treinado_reforcado.shape, y_treinado_reforcado.shape)
display(X_treinado_reforcado.shape, y_treinado_reforcado.shape)

Random Forest Classifier

In [ ]:
modelo2 = RandomForestClassifier(
	min_samples_split=16,
	min_samples_leaf=4,
	max_leaf_nodes=64,
	n_estimators=150,
	random_state=200,
	max_depth=32,
	class_weight="balanced_subsample",
	criterion="entropy",
	n_jobs=-1,
)

modelo2.fit(X_treinado_reforcado, y_treinado_reforcado)

y_pred_novo = modelo2.predict(X_test)

print(
	"Balanced Accuracy do novo modelo: ",
	balanced_accuracy_score(y_test, y_pred_novo) * 100,
	"%",
)

##### Grid Search

In [ ]:
# Grid Search
# 432 combinações
param_grid = {
	"min_samples_split": [4, 16, 32],
	"min_samples_leaf": [4, 16, 32, 64],
	"max_leaf_nodes": [4, 16, 32, 64],
	"n_estimators": [100, 150, 200],
	"max_depth": [4, 16, 32],
}

rf = RandomForestClassifier(
	random_state=200,
	class_weight="balanced_subsample",
	criterion="entropy",
)

scorer = make_scorer(balanced_accuracy_score)

grid_search = GridSearchCV(
	estimator=rf,
	param_grid=param_grid,
	scoring=scorer,
	cv=5,
	n_jobs=-1,
	verbose=2,
)

# fit
grid_search.fit(X_treinado_reforcado, y_treinado_reforcado)

# Best model and parameters
print("Best balanced accuracy:", grid_search.best_score_)
print("Best params:", grid_search.best_params_)

# Predict with best model
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
print("Test balanced accuracy:", balanced_accuracy_score(y_test, y_pred))

Print final (~ 50 minutos):
 * Fitting 5 folds for each of 432 candidates, totalling 2160 fits
 * Best balanced accuracy: 0.9039040507887066
 * Best params: {'max_depth': 16, 'max_leaf_nodes': 64, 'min_samples_leaf': 4, 'min_samples_split': 16, 'n_estimators': 200}
* Test balanced accuracy: 0.9663375492184703